# Data cleaning: HR roster and payroll export
Goal: read two messy breach files correctly, clean them with basic pandas, validate
key fields, and combine them into one list of people.
Data is synthetic (not real people). Inputs are read-only; full SSNs are never shown.


In [1]:
from pathlib import Path
import re
import pandas as pd
import openpyxl

In [2]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [13]:
ROOT = Path.cwd().parent                     # notebooks/ -> project root
DOCS = ROOT / "data" / "raw" / "corpus_v1" / "docs"
SHEETS = DOCS / "sheets"
HR = SHEETS / "hr_roster.xlsx"
PAYROLL = SHEETS / "payroll_export.csv"

In [14]:
print(HR.exists(), PAYROLL.exists())

True True


In [25]:
def shape(value):
    """123-45-6789 -> ddd-dd-dddd, Smith -> aaaaa."""
    if pd.isna(value):
        return "<NA>"
    return re.sub(r"\d", "d", re.sub(r"[A-Za-z]", "a", str(value)))

def mask(value):
    """Keep only the last 4 digits of anything number-like."""
    s = str(value)
    digits = re.sub(r"\D", "", s)
    return s if len(digits) < 6 else "***" + digits[-4:]



In [26]:
print(mask("123-45-6789"), mask("02139"), mask(None))

***6789 02139 None


In [27]:
print(shape("123-45-6789"), shape("APT6F"), shape(float("nan")))

ddd-dd-dddd aaada <NA>


In [29]:
hr_naive = pd.read_excel(HR)
print(hr_naive.shape)
hr_naive.head(7)

(63, 15)


,HARBORLINE LOGISTICS INC,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,Employee roster export - 2026-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SYNTHETIC - NOT REAL - test data only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Employee,NaN,NaN,NaN,NaN,NaN,NaN,Home Address,NaN,NaN,NaN,Contact,NaN,Hire Date,Department
3,ID,First,Middle,Last,Suffix,SSN,DOB,Street,City,State,ZIP,Phone,Email,NaN,NaN
4,429271,Matthew,NaN,Smith,NaN,703-69-7795,1960-05-27 00:00:00,3121 CARR VALLEY,Pittston,PA,18641,+17175550167,matthew.smith39@example.org,2025-02-18 00:00:00,Sales
5,732870,Shane,NaN,Henderson,NaN,250-19-0091,1972-02-23 00:00:00,710 W CESAR CHAVEZ ST,Austin,TX,78701,+15125550136,shane.henderson70@example.org,2020-05-02 00:00:00,Warehouse
6,826284,Timothy,NaN,Van Dyke,II,664-44-0208,1998-12-23 00:00:00,4603 ANDERSON KEYS,Orlando,FL,32825,+13055550128,timothy.vandyke31@example.com,2019-09-06 00:00:00,Operations


pandas used the first row (a company name, i.e. a title) as the header. The real column names are a few rows down.

In [30]:
pd.read_excel(HR, header=None).head(7)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,HARBORLINE LOGISTICS INC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Employee roster export - 2026-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SYNTHETIC - NOT REAL - test data only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Employee,NaN,NaN,NaN,NaN,NaN,NaN,Home Address,NaN,NaN,NaN,Contact,NaN,Hire Date,Department
4,ID,First,Middle,Last,Suffix,SSN,DOB,Street,City,State,ZIP,Phone,Email,NaN,NaN
5,429271,Matthew,NaN,Smith,NaN,703-69-7795,1960-05-27 00:00:00,3121 CARR VALLEY,Pittston,PA,18641,+17175550167,matthew.smith39@example.org,2025-02-18 00:00:00,Sales
6,732870,Shane,NaN,Henderson,NaN,250-19-0091,1972-02-23 00:00:00,710 W CESAR CHAVEZ ST,Austin,TX,78701,+15125550136,shane.henderson70@example.org,2020-05-02 00:00:00,Warehouse


row 4 holds the column name

In [31]:
hr = pd.read_excel(HR, header=4, dtype=str)
print(hr.shape)
hr.columns.tolist()

(59, 15)


['ID',
 'First',
 'Middle',
 'Last',
 'Suffix',
 'SSN',
 'DOB',
 'Street',
 'City',
 'State',
 'ZIP',
 'Phone',
 'Email',
 'Unnamed: 13',
 'Unnamed: 14']

column 13 and 14 name sits above row 4

In [32]:
hr = hr.rename(columns={"Unnamed: 13": "Hire Date", "Unnamed: 14": "Department"})
hr.head()

,ID,First,Middle,Last,Suffix,SSN,DOB,Street,City,State,ZIP,Phone,Email,Hire Date,Department
0,429271,Matthew,NaN,Smith,NaN,703-69-7795,1960-05-27 00:00:00,3121 CARR VALLEY,Pittston,PA,18641,+17175550167,matthew.smith39@example.org,2025-02-18 00:00:00,Sales
1,732870,Shane,NaN,Henderson,NaN,250-19-0091,1972-02-23 00:00:00,710 W CESAR CHAVEZ ST,Austin,TX,78701,+15125550136,shane.henderson70@example.org,2020-05-02 00:00:00,Warehouse
2,826284,Timothy,NaN,Van Dyke,II,664-44-0208,1998-12-23 00:00:00,4603 ANDERSON KEYS,Orlando,FL,32825,+13055550128,timothy.vandyke31@example.com,2019-09-06 00:00:00,Operations
3,314895,Philip,David,Cannon,NaN,477-57-7245,1966-04-05 00:00:00,303 GENESEE ST,Utica,NY,13501,+17165550144,philip.cannon15@example.net,2025-07-16 00:00:00,Operations
4,756272,Bryan,Matthew,Walker,NaN,820-78-3614,1968-12-10 00:00:00,1201 LEOPARD ST,Corpus Christi,TX,78408,+17135550134,bryan.walker14@example.net,2019-08-11 00:00:00,Fleet


In [33]:
print(hr[["ID", "City", "State", "ZIP"]].head(8))
print("Naive read ZIPs:", hr_naive.iloc[:, 10].head(8).tolist())

       ID            City State    ZIP
0  429271        Pittston    PA  18641
1  732870          Austin    TX  78701
2  826284         Orlando    FL  32825
3  314895           Utica    NY  13501
4  756272  Corpus Christi    TX  78408
5  780876       Cambridge    MA   2139
6  871906         Roanoke    VA  24016
7  950142         Buffalo    NY  14202
Naive read ZIPs: [nan, nan, nan, 'ZIP', 18641, 78701, 32825, 13501]


some ZIPs have only 4 digits (e.g. Cambridge, MA shows 2139). The leading zero was lost because the file stored ZIPs as numbers. Reading with dtype=str keeps them as text so I can fix them later.

In [34]:
hr.info()
hr.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          59 non-null     object
 1   First       58 non-null     object
 2   Middle      34 non-null     object
 3   Last        58 non-null     object
 4   Suffix      3 non-null      object
 5   SSN         57 non-null     object
 6   DOB         57 non-null     object
 7   Street      57 non-null     object
 8   City        57 non-null     object
 9   State       57 non-null     object
 10  ZIP         57 non-null     object
 11  Phone       57 non-null     object
 12  Email       57 non-null     object
 13  Hire Date   57 non-null     object
 14  Department  58 non-null     object
dtypes: object(15)
memory usage: 7.0+ KB


ID             0
First          1
Middle        25
Last           1
Suffix        56
SSN            2
DOB            2
Street         2
City           2
State          2
ZIP            2
Phone          2
Email          2
Hire Date      2
Department     1
dtype: int64

59 rows, but most columns have only 57 values, so 2 rows are not normal employees. Suffix and Middle are mostly empty, which is normal.

In [35]:
hr[hr["SSN"].isna()]

,ID,First,Middle,Last,Suffix,SSN,DOB,Street,City,State,ZIP,Phone,Email,Hire Date,Department
26,C90910,NaN,NaN,BRIGHTLINE OFFICE SUPPLY LLC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Contractor
58,Total employees,57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
print("Exact duplicate rows:", hr.duplicated().sum())
hr[hr.duplicated(keep=False)][["ID", "First", "Last"]]

Exact duplicate rows: 1


,ID,First,Last
43,770546,Renée,Lefèvre
44,770546,Renée,Lefèvre


found 1 exact duplicate row (the same employee twice, ID 770546). I looked at both copies before removing it.

In [37]:
before = len(hr)
hr = hr[~hr["ID"].str.contains("total", case=False, na=False)]   # drop the summary row
hr = hr.dropna(how="all")                                          # drop fully empty rows
hr = hr.drop_duplicates()                                          # drop exact duplicates
print(before, "->", len(hr))

59 -> 57


removed the total row, empty rows and the duplicate: 59 -> 57 rows

moved the contractor company out of the people list: 57 -> 56 people. Breach notices go to people, not companies.

In [38]:
org_pattern = r"\b(?:LLC|INC|CORP|SUPPLY|SERVICES)\b"
orgs = hr[hr["Last"].str.contains(org_pattern, case=False, na=False)]
print(orgs[["ID", "First", "Last"]])
hr = hr.drop(orgs.index)
print("Rows left:", len(hr))

        ID First                          Last
26  C90910   NaN  BRIGHTLINE OFFICE SUPPLY LLC
Rows left: 56


In [39]:
text_cols = ["First", "Middle", "Last", "Suffix", "Street", "City", "State", "Department"]
padded = {}
for c in text_cols:
    values = hr[c].dropna()
    padded[c] = (values != values.str.strip()).sum()
padded

{'First': 4,
 'Middle': 0,
 'Last': 4,
 'Suffix': 0,
 'Street': 0,
 'City': 0,
 'State': 0,
 'Department': 0}

4 first names and 4 last names have extra spaces (e.g. "  Susan "). Extra spaces stop names from matching across files.

In [40]:
for c in text_cols:
    hr[c] = hr[c].str.strip().str.upper()
hr[["First", "Last", "City"]].head()

,First,Last,City
0,MATTHEW,SMITH,PITTSTON
1,SHANE,HENDERSON,AUSTIN
2,TIMOTHY,VAN DYKE,ORLANDO
3,PHILIP,CANNON,UTICA
4,BRYAN,WALKER,CORPUS CHRISTI


trimmed the spaces and made all text uppercase, so the same name is always written the same way

In [41]:
print(hr["ZIP"].str.len().value_counts())
hr.loc[hr["ZIP"].str.len() != 5, ["City", "State", "ZIP"]]

ZIP
5    50
4     6
Name: count, dtype: int64


,City,State,ZIP
5,CAMBRIDGE,MA,2139
33,NEWARK,NJ,7102
35,JERSEY CITY,NJ,7302
41,BOSTON,MA,2116
47,TRENTON,NJ,8608
55,EAST SANDWICH,MA,2537


6 ZIPs have only 4 digits, and all of them are in MA or NJ, where ZIPs start with 0. So this is the lost leading zero problem, not random bad data.

In [42]:
hr["ZIP"] = hr["ZIP"].str.zfill(5)
print(hr["ZIP"].str.len().value_counts())
hr.loc[hr["State"].isin(["NJ", "MA"]), ["City", "State", "ZIP"]]

ZIP
5    56
Name: count, dtype: int64


,City,State,ZIP
5,CAMBRIDGE,MA,02139
33,NEWARK,NJ,07102
35,JERSEY CITY,NJ,07302
41,BOSTON,MA,02116
47,TRENTON,NJ,08608
55,EAST SANDWICH,MA,02537


In [43]:
hr["DOB"] = pd.to_datetime(hr["DOB"])
hr["Hire Date"] = pd.to_datetime(hr["Hire Date"])
print(hr[["DOB", "Hire Date"]].dtypes)
print("Oldest DOB:", hr["DOB"].min().date(), "| Youngest DOB:", hr["DOB"].max().date())

DOB          datetime64[ns]
Hire Date    datetime64[ns]
dtype: object
Oldest DOB: 1946-08-03 | Youngest DOB: 2002-12-15


changed DOB and Hire Date from text to real dates. Birth dates go from 1946 to 2002, which looks reasonable.

In [44]:
unit = hr["Street"].str.contains(r"\bAPT|\bUNIT|\bUPPER$|\bLOWER$", na=False)
hr.loc[unit, ["Street"]]

,Street
22,330 PARK BLVD UPPER
24,501 N OREGON ST APT6F
43,5441 CHARLES FOREST LOWER


In [45]:
bad_shape = ~hr["SSN"].str.fullmatch(r"\d{3}-\d{2}-\d{4}", na=False)
bad_area = hr["SSN"].str.startswith(("000", "666", "9"), na=False)
print("Bad SSN format:", bad_shape.sum())
print("Impossible SSN area:", bad_area.sum())
hr.loc[bad_area, ["ID", "Last"]].assign(SSN=hr.loc[bad_area, "SSN"].map(mask))

Bad SSN format: 0
Impossible SSN area: 1


,ID,Last,SSN
9,948583,CALDERON,***1766


all SSNs have the right format (ddd-dd-dddd), but 1 starts with 666, which is never issued. I flag it for review instead of changing it, because I can't know the correct number.

In [46]:
as_of = pd.Timestamp("2026-10-01")
future_dob = hr["DOB"] > as_of
too_old = (as_of - hr["DOB"]).dt.days / 365.25 > 120
bad_zip = ~hr["ZIP"].str.fullmatch(r"\d{5}", na=False)
print("Future DOB:", future_dob.sum(), "| Age > 120:", too_old.sum(), "| Bad ZIP:", bad_zip.sum())

Future DOB: 0 | Age > 120: 0 | Bad ZIP: 0


In [47]:
print(PAYROLL.read_text(encoding="latin-1")[:400])

# SYNTHETIC - NOT REAL
Emp No;Employee Name;SS#;Pay Date;Gross Pay;Net Pay;Bank Routing;Account No;Dept;Street;City;State;Zip
111280;Fowler, Edwin;653342648;09/15/2026;2972,82;2199,89;;;OPS;2461 BRYANT TERRACE;Greenfield;CA;93927
149027;Gray, Anthony;27123751;09/15/2026;1799,12;1331,35;;;SLS;5 WASHINGTON ST;Newark;NJ;07102
153315;Hernandez, Lee K;152562089;09/15/2026;1574,94;1165,46;;;FIN;828 I ST


the first line is a comment, the columns are separated by ; not commas, money uses decimal commas (2972,82) and SSNs have no dashes

In [48]:
try:
    pd.read_csv(PAYROLL)
except Exception as e:
    print(type(e).__name__, "-", e)

UnicodeDecodeError - 'utf-8' codec can't decode byte 0xe8 in position 3071: invalid continuation byte


In [49]:
pay = pd.read_csv(PAYROLL, sep=";", encoding="latin-1", skiprows=1, dtype=str)
print(pay.shape)
print(pay.columns.tolist())
pay.head()

(80, 13)
['Emp No', 'Employee Name', 'SS#', 'Pay Date', 'Gross Pay', 'Net Pay', 'Bank Routing', 'Account No', 'Dept', 'Street', 'City', 'State', 'Zip']


,Emp No,Employee Name,SS#,Pay Date,Gross Pay,Net Pay,Bank Routing,Account No,Dept,Street,City,State,Zip
0,111280,"Fowler, Edwin",653342648,09/15/2026,"2972,82","2199,89",NaN,NaN,OPS,2461 BRYANT TERRACE,Greenfield,CA,93927
1,149027,"Gray, Anthony",27123751,09/15/2026,"1799,12","1331,35",NaN,NaN,SLS,5 WASHINGTON ST,Newark,NJ,07102
2,153315,"Hernandez, Lee K",152562089,09/15/2026,"1574,94","1165,46",NaN,NaN,FIN,828 I ST,Sacramento,CA,95814
3,184271,"Wright, Scott J",304305913,09/15/2026,"2463,42","1822,93",NaN,NaN,OPS,120 N DUKE ST,Lancaster,PA,17602
4,224118,"Baker, Shawn V",627848667,09/15/2026,"4876,58","3608,67",088263982,21274846773,OPS,2090 KITTREDGE ST,Berkeley,CA,94704


In [50]:
print(pay.isna().sum())
print("Literal '#N/A' strings left:", (pay == "#N/A").sum().sum())

Emp No            0
Employee Name     0
SS#               0
Pay Date          0
Gross Pay         0
Net Pay           0
Bank Routing     50
Account No       53
Dept              6
Street            0
City              0
State             0
Zip               0
dtype: int64
Literal '#N/A' strings left: 0


pandas turned the #N/A values into missing values automatically, so 0 are left as text. Bank Routing and Account No are mostly empty, which makes sense because not everyone uses direct deposit.

In [51]:
pay["SS#"].str.len().value_counts()

SS#
9    72
8     8
Name: count, dtype: int64

72 SSNs have 9 digits and 8 have only 8 digits. Like the ZIPs, these lost their leading zero.

In [52]:
pay["SS#"] = pay["SS#"].str.zfill(9)
pay["SSN"] = pay["SS#"].str[:3] + "-" + pay["SS#"].str[3:5] + "-" + pay["SS#"].str[5:]
print(pay["SS#"].str.len().value_counts())
pay["SSN"].map(mask).head()

SS#
9    80
Name: count, dtype: int64


0    ***2648
1    ***3751
2    ***2089
3    ***5913
4    ***8667
Name: SSN, dtype: object

In [53]:
for c in ["Gross Pay", "Net Pay"]:
    pay[c] = pay[c].str.replace(",", ".", regex=False).astype(float)
pay[["Gross Pay", "Net Pay"]].describe()

,Gross Pay,Net Pay
count,80.000000,80.000000
mean,4356.359000,3223.705500
std,1489.326191,1102.101105
min,1574.940000,1165.460000
25%,3255.527500,2409.090000
50%,4374.790000,3237.345000
75%,5725.577500,4236.925000
max,6459.290000,4779.870000


changed the decimal commas to dots and converted pay to numbers. Gross pay ranges from 1574.94 to 6459.29, which looks reasonable.

In [54]:
names = pay["Employee Name"].str.split(",", n=1, expand=True)
pay["Last"] = names[0].str.strip().str.upper()
pay["FirstMiddle"] = names[1].str.strip().str.upper()
pay[["Employee Name", "Last", "FirstMiddle"]].head(3)

,Employee Name,Last,FirstMiddle
0,"Fowler, Edwin",FOWLER,EDWIN
1,"Gray, Anthony",GRAY,ANTHONY
2,"Hernandez, Lee K",HERNANDEZ,LEE K


split "Last, First Middle" into separate columns at the first comma

In [55]:
suffix_pattern = r"\s(JR|SR|II|III|IV|JJ)$"
suffix = pay["Last"].str.extract(suffix_pattern, expand=False)
print(suffix.value_counts())
pay["Suffix"] = suffix.replace({"JJ": "JR"})            # JJ is a typo for JR
pay["Last"] = pay["Last"].str.replace(suffix_pattern, "", regex=True)
pay.loc[pay["Suffix"].notna(), ["Employee Name", "Last", "Suffix"]]

Last
III    2
II     2
JJ     1
JR     1
Name: count, dtype: int64


,Employee Name,Last,Suffix
12,"Young JJ, Benjamin",YOUNG,JR
20,"Tucker III, Bradley C",TUCKER,III
30,"Van Dyke II, Timothy",VAN DYKE,II
52,"Young JR, Benjamin",YOUNG,JR
60,"Tucker III, Bradley C",TUCKER,III
70,"Van Dyke II, Timothy",VAN DYKE,II


some last names contain a suffix (Tucker III, Van Dyke II). I moved suffixes to their own column. "Young JJ" is a typo for JR, because the same person is "Young JR" in the other pay period.

In [56]:
parts = pay["FirstMiddle"].str.split(" ", n=1, expand=True)
pay["First"] = parts[0]
pay["Middle"] = parts[1]
pay[["First", "Middle", "Last", "Suffix"]].head()

,First,Middle,Last,Suffix
0,EDWIN,None,FOWLER,NaN
1,ANTHONY,None,GRAY,NaN
2,LEE,K,HERNANDEZ,NaN
3,SCOTT,J,WRIGHT,NaN
4,SHAWN,V,BAKER,NaN


split first and middle names. Middle is empty when the name has no middle initial.

In [57]:
print(pay["Emp No"].value_counts().value_counts())
print(pay["Pay Date"].value_counts())
print("Exact duplicate rows:", pay.duplicated().sum())

count
2    40
Name: count, dtype: int64
Pay Date
09/15/2026    40
09/30/2026    40
Name: count, dtype: int64
Exact duplicate rows: 0


In [58]:
pay["Pay Date"] = pd.to_datetime(pay["Pay Date"], format="%m/%d/%Y")
for c in ["Street", "City", "State"]:
    pay[c] = pay[c].str.strip().str.upper()
print("ZIP lengths:", pay["Zip"].str.len().value_counts().to_dict())
pay_people = pay.sort_values("Pay Date").drop_duplicates(subset="Emp No", keep="last")
print(len(pay), "rows ->", len(pay_people), "people")

ZIP lengths: {5: 80}
80 rows -> 40 people


In [59]:
hr_std = pd.DataFrame({
    "source": "hr_roster", "emp_id": hr["ID"], "first_name": hr["First"],
    "middle": hr["Middle"], "last_name": hr["Last"], "suffix": hr["Suffix"],
    "ssn": hr["SSN"], "street": hr["Street"], "city": hr["City"],
    "state": hr["State"], "zip": hr["ZIP"],
})
pay_std = pd.DataFrame({
    "source": "payroll", "emp_id": pay_people["Emp No"], "first_name": pay_people["First"],
    "middle": pay_people["Middle"], "last_name": pay_people["Last"], "suffix": pay_people["Suffix"],
    "ssn": pay_people["SSN"], "street": pay_people["Street"], "city": pay_people["City"],
    "state": pay_people["State"], "zip": pay_people["Zip"],
})
combined = pd.concat([hr_std, pay_std], ignore_index=True)
print(combined.shape)
combined["source"].value_counts()

(96, 11)


source
hr_roster    56
payroll      40
Name: count, dtype: int64

gave both files the same column names and stacked them: 56 from HR + 40 from payroll = 96 rows

In [60]:
both = hr_std.merge(pay_std, on="ssn", suffixes=("_hr", "_pay"))
print("People in both files:", len(both))
print("Only in HR:", len(hr_std) - len(both), "| Only in payroll:", len(pay_std) - len(both))
both[both["last_name_hr"] != both["last_name_pay"]][["last_name_hr", "last_name_pay"]]

People in both files: 40
Only in HR: 16 | Only in payroll: 0


,last_name_hr,last_name_pay


all 40 payroll people are also in the HR file, and their names match after cleaning. Without the SSN padding and the JJ -> JR fix, some of them would not have matched.

In [61]:
unique_people = combined.drop_duplicates(subset="ssn")
print(len(combined), "rows ->", len(unique_people), "unique people")

96 rows -> 56 unique people
